# GP Deep Kernel Learning Tutorial

Purpose: regression with engineered high-dimensional features and a GP head.


## Learning Roadmap

- Compare vanilla kernel GP vs deep-kernel GP on engineered features.
- Inspect calibration changes alongside RMSE/NLL.
- Visualize learned feature space structure.


In [ ]:
# Step 1: import models and reproducibility utilities
# Configure Python path for local package imports
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'gp' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import math
import time
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)
np.random.seed(42)

from deepuq.models import DeepKernelGaussianProcessRegressor, GaussianProcessRegressor, RBFKernel


In [ ]:
# Step 2: define calibration metrics helper
def regression_metrics(y_true, mean, var):
    y_true = y_true.reshape(-1)
    mean = mean.reshape(-1)
    var = var.reshape(-1).clamp_min(1e-8)
    rmse = torch.sqrt(torch.mean((mean - y_true) ** 2)).item()
    nll = 0.5 * torch.mean(torch.log(2 * torch.pi * var) + (y_true - mean) ** 2 / var).item()
    std = torch.sqrt(var)
    lower = mean - 1.96 * std
    upper = mean + 1.96 * std
    coverage95 = torch.mean(((y_true >= lower) & (y_true <= upper)).float()).item()
    width95 = torch.mean(upper - lower).item()
    return {
        'rmse': rmse,
        'nll': nll,
        'coverage95': coverage95,
        'interval_width95': width95,
    }


In [ ]:
# Step 3: build high-dimensional engineered features
z_train = torch.linspace(-3.0, 3.0, 140).unsqueeze(-1)
base_fn = lambda z: torch.sin(1.4 * z) + 0.2 * z

# Engineered high-dimensional features
x_train = torch.cat([
    z_train,
    z_train**2,
    torch.sin(2 * z_train),
    torch.cos(3 * z_train),
    torch.exp(-0.4 * z_train**2),
], dim=1)
y_train = base_fn(z_train) + 0.10 * torch.randn_like(z_train)

z_test = torch.linspace(-4.5, 4.5, 320).unsqueeze(-1)
x_test = torch.cat([
    z_test,
    z_test**2,
    torch.sin(2 * z_test),
    torch.cos(3 * z_test),
    torch.exp(-0.4 * z_test**2),
], dim=1)
y_test = base_fn(z_test)


In [ ]:
# Step 4: train baseline GP and DKL-GP, then compare metrics
baseline = GaussianProcessRegressor(
    kernel=RBFKernel(lengthscale=torch.tensor([1.0] * x_train.shape[1]), outputscale=1.0),
    noise=0.01,
)
baseline.fit(x_train, y_train)
b_uq = baseline.predict_uq(x_test)

# DKL-GP
dkl = DeepKernelGaussianProcessRegressor(feature_dim=16, hidden_dims=(64, 64), epochs=220, lr=1e-3)
dkl.fit(x_train, y_train)
d_uq = dkl.predict_uq(x_test)

print('Baseline:', {k: round(v, 4) for k, v in regression_metrics(y_test.squeeze(-1), b_uq.mean, b_uq.total_var).items()})
print('DKL-GP  :', {k: round(v, 4) for k, v in regression_metrics(y_test.squeeze(-1), d_uq.mean, d_uq.total_var).items()})


In [ ]:
# Step 5: visualize predictive bands side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, uq, title in [
    (axes[0], b_uq, 'Vanilla ARD RBF GP'),
    (axes[1], d_uq, 'Deep Kernel GP'),
]:
    std = torch.sqrt(uq.total_var)
    ax.scatter(z_train.numpy(), y_train.numpy(), s=11, alpha=0.35, label='train')
    ax.plot(z_test.numpy(), y_test.numpy(), 'k--', lw=1.1, label='truth')
    ax.plot(z_test.numpy(), uq.mean.numpy(), lw=2, label='mean')
    ax.fill_between(
        z_test.squeeze(-1).numpy(),
        (uq.mean - 1.96 * std).numpy(),
        (uq.mean + 1.96 * std).numpy(),
        alpha=0.2,
    )
    ax.set_title(title)
    ax.legend(loc='best')
plt.tight_layout()
plt.show()


In [ ]:
# Additional diagnostic: calibration scatter and learned feature map
# We compare predicted std against absolute error for both models.
b_std = torch.sqrt(b_uq.total_var)
d_std = torch.sqrt(d_uq.total_var)
b_err = torch.abs(b_uq.mean - y_test.squeeze(-1))
d_err = torch.abs(d_uq.mean - y_test.squeeze(-1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(b_std.numpy(), b_err.numpy(), s=10, alpha=0.4, label='baseline')
axes[0].scatter(d_std.numpy(), d_err.numpy(), s=10, alpha=0.4, label='DKL')
axes[0].set_title('Std vs absolute error')
axes[0].set_xlabel('predicted std')
axes[0].set_ylabel('absolute error')
axes[0].legend(loc='best')

with torch.no_grad():
    feat = dkl.feature_extractor(x_train)
feat_np = feat[:, :2].detach().numpy()
axes[1].scatter(feat_np[:, 0], feat_np[:, 1], c=y_train.squeeze(-1).numpy(), cmap='viridis', s=18)
axes[1].set_title('First 2 learned DKL features (train)')
axes[1].set_xlabel('feature 1')
axes[1].set_ylabel('feature 2')
plt.tight_layout()
plt.show()
